# Домашнє завдання 7 - Часові ряди та JSONB у задачах ML

In [11]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow requests numpy
import re
from pathlib import Path
from urllib.request import urlretrieve
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pgserver
from sqlalchemy import create_engine, text
from IPython.display import display, Markdown

pg = pgserver.get_server('/tmp/hw7_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)
with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

def execute_script(sql):
    raw = engine.raw_connection()
    try:
        with raw.cursor() as cur:
            cur.execute(sql)
        raw.commit()
    finally:
        raw.close()

def run_query(sql):
    return pd.read_sql(text(sql), engine)

def scalar(sql):
    with engine.connect() as conn:
        return conn.execute(text(sql)).scalar_one()

def explain(sql):
    return run_query('EXPLAIN (ANALYZE, BUFFERS)\n' + sql)

def plan_text(df):
    return '\n'.join(df.iloc[:, 0].astype(str).tolist())

def exec_ms(df):
    m = re.search(r'Execution Time:\s*([0-9.]+)\s*ms', plan_text(df))
    return float(m.group(1)) if m else None

def plan_nodes(df):
    txt = plan_text(df)
    names = ['Bitmap Index Scan','Bitmap Heap Scan','Index Scan','Index Only Scan','Seq Scan','HashAggregate','GroupAggregate','WindowAgg','Sort']
    return [name for name in names if name in txt]

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


## 1. Setup і завантаження NYC Taxi

Офіційне джерело: NYC TLC Yellow Taxi Trip Records. Notebook завантажує March 2024 Parquet. Щоб зберегти покриття всього місяця, sample береться з рівномірно розподілених Parquet row groups, а не через простий `head()`. Якщо official file недоступний, автоматично використовується deterministic stub 60 days x 5 zones = 300 rows.

In [12]:
URL = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-03.parquet'
parquet_path = Path('/content/yellow_tripdata_2024-03.parquet')
columns = ['tpep_pickup_datetime','tpep_dropoff_datetime','PULocationID','DOLocationID','trip_distance','fare_amount','passenger_count']
USE_STUB = False
DATA_MODE = None

try:
    if not parquet_path.exists():
        urlretrieve(URL, parquet_path)
    pf = pq.ParquetFile(parquet_path)
    group_ids = sorted(set(np.linspace(0, pf.num_row_groups - 1, min(32, pf.num_row_groups), dtype=int).tolist()))
    per_group = max(1, 500_000 // len(group_ids))
    parts = []
    for rg in group_ids:
        chunk = pf.read_row_group(rg, columns=columns).to_pandas()
        if len(chunk) > per_group:
            chunk = chunk.sample(per_group, random_state=42 + rg)
        parts.append(chunk)
    df = pd.concat(parts, ignore_index=True)
    df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 500)]
    df = df[(df['trip_distance'] > 0) & (df['trip_distance'] < 100)]
    df = df[df['PULocationID'].notna()].copy()
    if df.empty:
        raise ValueError('Empty sample after cleaning')
    df.to_sql('nyc_taxi_raw', engine, if_exists='replace', index=False, chunksize=10_000)
    DATA_MODE = 'official NYC TLC Parquet sample'
    print('Raw rows:', len(df))
    print('Pickup range:', df['tpep_pickup_datetime'].min(), '->', df['tpep_pickup_datetime'].max())
except Exception as exc:
    USE_STUB = True
    DATA_MODE = 'deterministic stub fallback'
    print('Official dataset unavailable, using stub fallback:', type(exc).__name__, str(exc)[:250])

if not USE_STUB:
    execute_script(r'''
    DROP VIEW IF EXISTS v_t7_hw_taxi_features CASCADE;
    DROP TABLE IF EXISTS t7_hw_taxi_daily CASCADE;
    CREATE TABLE t7_hw_taxi_daily AS
    SELECT
        "PULocationID"::INTEGER AS pickup_zone_id,
        DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
        COUNT(*)::INTEGER AS trip_count,
        AVG(fare_amount)::NUMERIC(10,2) AS avg_fare,
        SUM(trip_distance)::NUMERIC(12,2) AS total_distance
    FROM nyc_taxi_raw
    WHERE fare_amount > 0 AND fare_amount < 500
      AND trip_distance > 0 AND trip_distance < 100
      AND "PULocationID" IS NOT NULL
    GROUP BY "PULocationID"::INTEGER,
             DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;
    ALTER TABLE t7_hw_taxi_daily ADD PRIMARY KEY (pickup_zone_id, ts_day);
    ''')
else:
    execute_script(r'''
    DROP VIEW IF EXISTS v_t7_hw_taxi_features CASCADE;
    DROP TABLE IF EXISTS t7_hw_taxi_daily CASCADE;
    CREATE TABLE t7_hw_taxi_daily (
        pickup_zone_id INTEGER NOT NULL,
        ts_day DATE NOT NULL,
        trip_count INTEGER NOT NULL CHECK (trip_count >= 0),
        avg_fare NUMERIC(10,2) NOT NULL CHECK (avg_fare >= 0),
        total_distance NUMERIC(12,2) NOT NULL CHECK (total_distance >= 0),
        PRIMARY KEY (pickup_zone_id, ts_day)
    );
    INSERT INTO t7_hw_taxi_daily
    SELECT
        z.zone_id,
        DATE '2024-03-31' - (59 - n),
        500 + n*7 + z.zone_id*50 + (n%5)*30,
        (12.5 + z.zone_id*0.7 + (n%4)*0.4)::NUMERIC(10,2),
        (1500 + n*25 + z.zone_id*120)::NUMERIC(12,2)
    FROM generate_series(0,59) AS n
    CROSS JOIN (VALUES (1),(2),(3),(4),(5)) AS z(zone_id);
    ''')

print('DATA_MODE:', DATA_MODE)
display(run_query('SELECT COUNT(*) n_rows, COUNT(DISTINCT pickup_zone_id) n_zones, MIN(ts_day) min_day, MAX(ts_day) max_day FROM t7_hw_taxi_daily'))

Raw rows: 472759
Pickup range: 2024-02-29 23:04:32 -> 2024-03-31 23:59:56
DATA_MODE: official NYC TLC Parquet sample


,n_rows,n_zones,min_day,max_day
0,5651,246,2024-02-29,2024-03-31


### Production-style демо TIMESTAMPTZ
Часові мітки TLC є локальними значеннями часу Нью-Йорка без timezone. Для production-подій AT TIME ZONE 'America/New_York' перетворює їх на абсолютні значення типу TIMESTAMPTZ. У цьому домашньому завданні денна агрегація залишається у вигляді локальної для Нью-Йорка дати типу DATE.

In [13]:
if not USE_STUB:
    execute_script(r'''
    DROP TABLE IF EXISTS t7_hw_taxi_trips_typed CASCADE;
    CREATE TABLE t7_hw_taxi_trips_typed AS
    SELECT
        "PULocationID"::INTEGER AS pickup_zone_id,
        "DOLocationID"::INTEGER AS dropoff_zone_id,
        (tpep_pickup_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS pickup_at,
        (tpep_dropoff_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS dropoff_at,
        trip_distance::NUMERIC(10,2) AS trip_distance,
        fare_amount::NUMERIC(10,2) AS fare_amount,
        passenger_count::INTEGER AS passenger_count
    FROM nyc_taxi_raw
    WHERE fare_amount > 0 AND fare_amount < 500
      AND trip_distance > 0 AND trip_distance < 100;
    ''')
    display(run_query('SELECT pickup_at, pg_typeof(pickup_at) pickup_type FROM t7_hw_taxi_trips_typed LIMIT 5'))
else:
    print('TIMESTAMPTZ demo skipped on stub fallback')

,pickup_at,pickup_type
0,2024-03-05 22:56:49+00:00,timestamp with time zone
1,2024-03-09 23:20:37+00:00,timestamp with time zone
2,2024-03-10 07:37:01+00:00,timestamp with time zone
3,2024-03-08 21:24:23+00:00,timestamp with time zone
4,2024-03-07 17:05:08+00:00,timestamp with time zone


## 2. Time-series ML features

Усі window definitions мають `PARTITION BY pickup_zone_id`. Lag features використовують 1, 7 і 30 previous rows. Rolling 7-day window використовує `INTERVAL '6 days'`, rolling 30-day window - `INTERVAL '29 days'`. Expanding window починається з `UNBOUNDED PRECEDING`.

In [14]:
execute_script(r'''
CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,
        LAG(trip_count,1) OVER w AS trips_lag_1d,
        LAG(trip_count,7) OVER w AS trips_lag_7d,
        LAG(trip_count,30) OVER w AS trips_lag_30d,
        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips
    FROM t7_hw_taxi_daily
    WINDOW
        w AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),
        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
        ),
        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
        ),
        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
)
SELECT *
FROM features_basic
WHERE trips_lag_30d IS NOT NULL;
''')

display(run_query('SELECT * FROM v_t7_hw_taxi_features ORDER BY pickup_zone_id, ts_day LIMIT 20'))
print('Feature rows:', scalar('SELECT COUNT(*) FROM v_t7_hw_taxi_features'))

,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
0,4,2024-03-31,81,17.96,244.02,103,75,37,18.115714,2.127517,1414,45.612903
1,7,2024-03-31,22,25.28,108.69,21,22,9,19.981429,4.654655,387,12.483871
2,10,2024-03-31,8,58.86,99.23,6,7,9,60.922857,14.024195,163,5.258065
3,12,2024-03-31,9,25.24,44.74,5,6,3,22.875714,5.538205,161,5.193548
4,13,2024-03-31,49,25.24,269.46,73,57,97,22.711429,1.912032,2989,96.419355
5,17,2024-03-31,9,27.77,49.00,3,9,8,24.458571,5.990945,174,5.612903
6,24,2024-03-31,47,14.53,120.80,40,33,50,15.961429,1.638679,1292,41.677419
7,25,2024-03-31,18,28.14,79.14,10,18,8,23.041429,6.029897,339,10.935484
8,33,2024-03-31,15,24.29,61.62,15,16,9,27.078571,4.547134,456,14.709677
9,37,2024-03-31,13,23.40,63.54,12,6,1,30.878571,8.063309,197,6.354839


Feature rows: 95


### Calendar-spine
Calendar-spine через `generate_series` і `LEFT JOIN` показує, як явно зберегти дні без подій.

In [15]:
display(run_query(r'''
WITH bounds AS (
    SELECT MIN(ts_day) min_day, MAX(ts_day) max_day
    FROM t7_hw_taxi_daily
),
target_zone AS (
    SELECT pickup_zone_id
    FROM t7_hw_taxi_daily
    GROUP BY pickup_zone_id
    ORDER BY COUNT(*) DESC, pickup_zone_id
    LIMIT 1
),
spine AS (
    SELECT generate_series(
        (SELECT min_day FROM bounds),
        (SELECT max_day FROM bounds),
        INTERVAL '1 day'
    )::DATE AS d
)
SELECT
    z.pickup_zone_id,
    s.d AS calendar_day,
    COALESCE(m.trip_count,0) AS trip_count_filled
FROM target_zone z
CROSS JOIN spine s
LEFT JOIN t7_hw_taxi_daily m
    ON m.pickup_zone_id = z.pickup_zone_id
   AND m.ts_day = s.d
ORDER BY s.d
LIMIT 20;
'''))

,pickup_zone_id,calendar_day,trip_count_filled
0,48,2024-02-29,1
1,48,2024-03-01,306
2,48,2024-03-02,453
3,48,2024-03-03,361
4,48,2024-03-04,250
5,48,2024-03-05,358
6,48,2024-03-06,404
7,48,2024-03-07,334
8,48,2024-03-08,341
9,48,2024-03-09,513


## 3. JSONB event-log

`t7_hw_events` містить 50 000 synthetic events і 5 event types: `page_view`, `click`, `add_to_cart`, `checkout`, `recommendation_shown`. Кожен тип має іншу структуру JSONB payload.

In [16]:
execute_script(r'''
DROP TABLE IF EXISTS t7_hw_checkouts_norm CASCADE;
DROP TABLE IF EXISTS t7_hw_events CASCADE;
CREATE TABLE t7_hw_events (
    event_id INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id INTEGER NOT NULL,
    event_ts TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type TEXT NOT NULL,
    payload JSONB NOT NULL
);

INSERT INTO t7_hw_events (user_id,event_ts,event_type,payload)
SELECT
    (n%1000)+1,
    TIMESTAMPTZ '2024-04-01 00:00:00+00' - ((n%60)||' days')::INTERVAL,
    CASE n%5
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE 'recommendation_shown'
    END,
    CASE n%5
        WHEN 0 THEN jsonb_build_object(
            'url','/product/'||((n%200)+1),
            'referrer',(ARRAY['google','direct','email','fb'])[(n%4)+1],
            'device',(ARRAY['mobile','desktop','tablet'])[(n%3)+1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id','btn_'||((n%50)+1),
            'page_url','/category/'||((n%20)+1),
            'device',(ARRAY['mobile','desktop'])[(n%2)+1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id',(n%200)+1,
            'price',((n%100)+10)::NUMERIC(10,2),
            'quantity',(n%5)+1,
            'tags',jsonb_build_array((ARRAY['promo','new','clearance'])[(n%3)+1])
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id','o_'||(n+100000),
            'total',((n%500)+50)::NUMERIC(10,2),
            'items_count',(n%10)+1,
            'payment_method',(ARRAY['card','paypal','crypto'])[(n%3)+1]
        )
        ELSE jsonb_build_object(
            'model_name','rec_v'||((n%5)+1),
            'model_version','1.'||(n%10),
            'recommendations',jsonb_build_array((n%200)+1,((n+1)%200)+1,((n+2)%200)+1),
            'experiment','exp_'||((n%3)+1)
        )
    END
FROM generate_series(1,50000) AS n;
ANALYZE t7_hw_events;
''')
display(run_query('SELECT event_type, COUNT(*) n_events FROM t7_hw_events GROUP BY event_type ORDER BY event_type'))

,event_type,n_events
0,add_to_cart,10000
1,checkout,10000
2,click,10000
3,page_view,10000
4,recommendation_shown,10000


### JSONB operators and JSONPath
Демонструються containment `@>`, key existence `?`, any-key `?|`, `jsonb_path_query` і `jsonb_path_exists`.

In [17]:
display(run_query(r'''
SELECT event_id,user_id,event_ts,payload
FROM t7_hw_events
WHERE event_type='checkout'
  AND payload @> jsonb_build_object('payment_method','card')
LIMIT 10;
'''))

display(run_query(r'''
SELECT event_id,event_type,payload
FROM t7_hw_events
WHERE payload ? 'tags'
LIMIT 10;
'''))

display(run_query(r'''
SELECT event_id,event_type,payload
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment','model_version']
LIMIT 10;
'''))

display(run_query(r'''
SELECT event_id, jsonb_path_query(payload,'$.recommendations[*]') AS recommended_pid
FROM t7_hw_events
WHERE event_type='recommendation_shown'
LIMIT 15;
'''))

display(run_query(r'''
SELECT event_id, payload->>'order_id' AS order_id, (payload->>'total')::NUMERIC(10,2) AS total
FROM t7_hw_events
WHERE event_type='checkout'
  AND jsonb_path_exists(payload,'$.total ? (@ > 300)')
LIMIT 10;
'''))

,event_id,user_id,event_ts,payload
0,3,4,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items..."
1,18,19,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items..."
2,33,34,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items..."
3,48,49,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items..."
4,63,64,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'item..."
5,78,79,2024-03-14 00:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'item..."
6,93,94,2024-02-28 00:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'item..."
7,108,109,2024-02-13 00:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'item..."
8,123,124,2024-03-29 00:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'item..."
9,138,139,2024-03-14 00:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'item..."


,event_id,event_type,payload
0,2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quanti..."
1,7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3..."
2,12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity':..."
3,17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quanti..."
4,22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3..."
5,27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity':..."
6,32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quanti..."
7,37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3..."
8,42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity':..."
9,47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quanti..."


,event_id,event_type,payload
0,4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
1,9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
2,14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
3,19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
4,24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
5,29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
6,34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
7,39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
8,44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
9,49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."


,event_id,recommended_pid
0,4,5
1,4,6
2,4,7
3,9,10
4,9,11
5,9,12
6,14,15
7,14,16
8,14,17
9,19,20


,event_id,order_id,total
0,253,o_100253,303.0
1,258,o_100258,308.0
2,263,o_100263,313.0
3,268,o_100268,318.0
4,273,o_100273,323.0
5,278,o_100278,328.0
6,283,o_100283,333.0
7,288,o_100288,338.0
8,293,o_100293,343.0
9,298,o_100298,348.0


## 4. GIN-index demo + EXPLAIN ANALYZE

Зберігаються 4 required plans: containment до/після GIN та path equality до/після expression B-tree index.

In [18]:
execute_script(r'''
DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
DROP INDEX IF EXISTS idx_t7_events_device_btree;
ANALYZE t7_hw_events;
''')

containment_sql = "SELECT COUNT(*) FROM t7_hw_events WHERE payload @> jsonb_build_object('payment_method','card');"
explain_containment_before = explain(containment_sql)
display(explain_containment_before)

execute_script(r'''
CREATE INDEX idx_t7_events_payload_gin_pathops
    ON t7_hw_events USING GIN (payload jsonb_path_ops);
ANALYZE t7_hw_events;
''')

display(run_query("SELECT pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops')) AS gin_index_size"))
explain_containment_after = explain(containment_sql)
display(explain_containment_after)

device_sql = "SELECT COUNT(*) FROM t7_hw_events WHERE payload ->> 'device' = 'mobile';"
explain_device_before = explain(device_sql)
display(explain_device_before)

execute_script(r'''
CREATE INDEX idx_t7_events_device_btree
    ON t7_hw_events ((payload ->> 'device'));
ANALYZE t7_hw_events;
''')

explain_device_after = explain(device_sql)
display(explain_device_after)

plan_summary = pd.DataFrame([
    ['containment_before_gin', exec_ms(explain_containment_before), ', '.join(plan_nodes(explain_containment_before))],
    ['containment_after_gin', exec_ms(explain_containment_after), ', '.join(plan_nodes(explain_containment_after))],
    ['device_before_btree', exec_ms(explain_device_before), ', '.join(plan_nodes(explain_device_before))],
    ['device_after_btree', exec_ms(explain_device_after), ', '.join(plan_nodes(explain_device_after))],
], columns=['scenario','execution_ms','plan_nodes'])
display(plan_summary)

,QUERY PLAN
0,Aggregate (cost=1755.75..1755.76 rows=1 width...
1,Buffers: shared hit=1000
2,-> Seq Scan on t7_hw_events (cost=0.00..17...
3,Filter: (payload @> jsonb_build_object...
4,Rows Removed by Filter: 46666
5,Buffers: shared hit=1000
6,Planning:
7,Buffers: shared hit=5
8,Planning Time: 0.162 ms
9,Execution Time: 46.264 ms


,gin_index_size
0,1144 kB


,QUERY PLAN
0,Aggregate (cost=1084.06..1084.07 rows=1 width...
1,Buffers: shared hit=1004
2,-> Bitmap Heap Scan on t7_hw_events (cost=...
3,Recheck Cond: (payload @> jsonb_build_...
4,Heap Blocks: exact=1000
5,Buffers: shared hit=1004
6,-> Bitmap Index Scan on idx_t7_events...
7,Index Cond: (payload @> jsonb_bu...
8,Buffers: shared hit=4
9,Planning:


,QUERY PLAN
0,Aggregate (cost=1750.62..1750.63 rows=1 width...
1,Buffers: shared hit=1000
2,-> Seq Scan on t7_hw_events (cost=0.00..17...
3,Filter: ((payload ->> 'device'::text) ...
4,Rows Removed by Filter: 41667
5,Buffers: shared hit=1000
6,Planning Time: 0.080 ms
7,Execution Time: 12.118 ms


,QUERY PLAN
0,Aggregate (cost=1243.20..1243.21 rows=1 width...
1,Buffers: shared hit=1000 read=9
2,-> Bitmap Heap Scan on t7_hw_events (cost=...
3,Recheck Cond: ((payload ->> 'device'::...
4,Heap Blocks: exact=1000
5,Buffers: shared hit=1000 read=9
6,-> Bitmap Index Scan on idx_t7_events...
7,Index Cond: ((payload ->> 'devic...
8,Buffers: shared read=9
9,Planning:


,scenario,execution_ms,plan_nodes
0,containment_before_gin,46.264,Seq Scan
1,containment_after_gin,6.593,"Bitmap Index Scan, Bitmap Heap Scan, Index Scan"
2,device_before_btree,12.118,Seq Scan
3,device_after_btree,3.255,"Bitmap Index Scan, Bitmap Heap Scan, Index Scan"


## 5. JSONB vs normalized checkout table

Для одного event type створюється typed normalized table і порівнюється той самий aggregate через `EXPLAIN (ANALYZE, BUFFERS)`.

In [19]:
execute_script(r'''
DROP TABLE IF EXISTS t7_hw_checkouts_norm CASCADE;
CREATE TABLE t7_hw_checkouts_norm (
    event_id INTEGER PRIMARY KEY,
    user_id INTEGER NOT NULL,
    event_ts TIMESTAMPTZ NOT NULL,
    order_id TEXT NOT NULL,
    total NUMERIC(10,2) NOT NULL,
    items_count INTEGER NOT NULL CHECK (items_count > 0),
    payment_method TEXT NOT NULL CHECK (payment_method IN ('card','paypal','crypto'))
);
INSERT INTO t7_hw_checkouts_norm
SELECT
    event_id,
    user_id,
    event_ts,
    payload->>'order_id',
    (payload->>'total')::NUMERIC(10,2),
    (payload->>'items_count')::INTEGER,
    payload->>'payment_method'
FROM t7_hw_events
WHERE event_type='checkout';
CREATE INDEX idx_t7_checkouts_norm_pm ON t7_hw_checkouts_norm(payment_method);
ANALYZE t7_hw_checkouts_norm;
''')

jsonb_agg_sql = r'''
SELECT payload->>'payment_method' AS payment_method,
       AVG((payload->>'total')::NUMERIC) AS avg_total,
       COUNT(*) AS n
FROM t7_hw_events
WHERE event_type='checkout'
GROUP BY payload->>'payment_method'
ORDER BY avg_total DESC
LIMIT 3;
'''

normalized_agg_sql = r'''
SELECT payment_method, AVG(total) AS avg_total, COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3;
'''

explain_jsonb_agg = explain(jsonb_agg_sql)
explain_norm_agg = explain(normalized_agg_sql)
display(explain_jsonb_agg)
display(explain_norm_agg)
display(run_query(jsonb_agg_sql))
display(run_query(normalized_agg_sql))

,QUERY PLAN
0,Limit (cost=1958.19..1958.20 rows=3 width=72)...
1,Buffers: shared hit=1000
2,-> Sort (cost=1958.19..1972.20 rows=5604 w...
3,Sort Key: (avg(((payload ->> 'total'::...
4,Sort Method: quicksort Memory: 25kB
5,Buffers: shared hit=1000
6,-> HashAggregate (cost=1801.70..1885...
7,Group Key: (payload ->> 'payment...
8,Batches: 1 Memory Usage: 217kB
9,Buffers: shared hit=1000


,QUERY PLAN
0,Limit (cost=269.06..269.07 rows=3 width=46) (...
1,Buffers: shared hit=94
2,-> Sort (cost=269.06..269.07 rows=3 width=...
3,Sort Key: (avg(total)) DESC
4,Sort Method: quicksort Memory: 25kB
5,Buffers: shared hit=94
6,-> HashAggregate (cost=269.00..269.0...
7,Group Key: payment_method
8,Batches: 1 Memory Usage: 24kB
9,Buffers: shared hit=94


,payment_method,avg_total,n
0,paypal,300.524752,3333
1,card,300.500000,3334
2,crypto,300.475248,3333


,payment_method,avg_total,n
0,paypal,300.524752,3333
1,card,300.500000,3334
2,crypto,300.475248,3333


## 6. Reflection

In [20]:
def fm(v):
    return f'{v:.3f} ms' if v is not None else 'not parsed'

reflection = f'''
### Reflection

У моєму запуску containment-запит `payload @> ...` до GIN мав execution time **{fm(exec_ms(explain_containment_before))}**, а після створення `jsonb_path_ops` GIN - **{fm(exec_ms(explain_containment_after))}**. До індексу план містив **{', '.join(plan_nodes(explain_containment_before)) or 'no automatically detected node'}**, після індексу - **{', '.join(plan_nodes(explain_containment_after)) or 'no automatically detected node'}**. Якщо planner залишив `Seq Scan`, це не є помилкою: для 50 000 rows або низької селективності повний scan іноді оцінюється дешевше за index lookup.

GIN добре відповідає containment та JSONPath workload, бо індексує структуру JSONB. Але `payload ->> 'device' = 'mobile'` є path-specific text equality, тому expression B-tree index точніше відповідає predicate. До нього execution time був **{fm(exec_ms(explain_device_before))}**, після - **{fm(exec_ms(explain_device_after))}**.

GIN створює read/write trade-off: читання за `@>` може прискоритися, але INSERT/UPDATE/DELETE стають дорожчими, а сам індекс займає storage. Тому індекс треба обирати під реальний workload.

Для ML metadata, model registry та event attributes я б обрав JSONB, якщо schema частково змінна і потрібні SQL, ACID та JOIN. Для стабільних полів, які часто використовуються у filtering, aggregation або constraints, normalized typed table природніша. У моєму запуску JSONB aggregation виконувалась за **{fm(exec_ms(explain_jsonb_agg))}**, normalized aggregation - за **{fm(exec_ms(explain_norm_agg))}**; висновок про швидкість потрібно робити саме з фактичних планів.

Новий event-type `refund` у JSONB можна додати без `ALTER TABLE`, визначивши payload schema та validation. У normalized design зазвичай потрібна нова typed table або schema migration. Це основний trade-off між schema flexibility та сильнішою структурою даних.
'''

display(Markdown(reflection))


### Reflection

У моєму запуску containment-запит `payload @> ...` до GIN мав execution time **46.264 ms**, а після створення `jsonb_path_ops` GIN - **6.593 ms**. До індексу план містив **Seq Scan**, після індексу - **Bitmap Index Scan, Bitmap Heap Scan, Index Scan**. Якщо planner залишив `Seq Scan`, це не є помилкою: для 50 000 rows або низької селективності повний scan іноді оцінюється дешевше за index lookup.

GIN добре відповідає containment та JSONPath workload, бо індексує структуру JSONB. Але `payload ->> 'device' = 'mobile'` є path-specific text equality, тому expression B-tree index точніше відповідає predicate. До нього execution time був **12.118 ms**, після - **3.255 ms**.

GIN створює read/write trade-off: читання за `@>` може прискоритися, але INSERT/UPDATE/DELETE стають дорожчими, а сам індекс займає storage. Тому індекс треба обирати під реальний workload.

Для ML metadata, model registry та event attributes я б обрав JSONB, якщо schema частково змінна і потрібні SQL, ACID та JOIN. Для стабільних полів, які часто використовуються у filtering, aggregation або constraints, normalized typed table природніша. У моєму запуску JSONB aggregation виконувалась за **14.484 ms**, normalized aggregation - за **4.112 ms**; висновок про швидкість потрібно робити саме з фактичних планів.

Новий event-type `refund` у JSONB можна додати без `ALTER TABLE`, визначивши payload schema та validation. У normalized design зазвичай потрібна нова typed table або schema migration. Це основний trade-off між schema flexibility та сильнішою структурою даних.
